# 04 - Iterators, Generators, Decorators & Type Annotations

**Sources:** Bootcamp BP3 (section headings only: *Iterators, Generators, Decorators, Type
Annotations* - the body was empty in the PDF, so this notebook fills it in) and Summer School
*CursP_BasePython* (decorators as function wrappers, `debug` decorator, stacking order).

**Contents:** iteration protocol -> writing iterators -> generators -> lazy pipelines ->
decorators (simple, with arguments, class-based, stacking) -> type annotations and `mypy`.

## 1. The iteration protocol

* An **iterable** has `__iter__()` returning an iterator (lists, strings, dicts, files, ranges, ...).
* An **iterator** has `__next__()` (returns the next item or raises `StopIteration`) and `__iter__()` returning itself.
* A `for` loop is sugar for: `it = iter(obj)`, then `next(it)` until `StopIteration`.

In [1]:
numbers = [10, 20, 30]
it = iter(numbers)
print(next(it), next(it), next(it))
try:
    next(it)
except StopIteration:
    print("exhausted")

# What `for x in numbers: ...` really does
it = iter(numbers)
while True:
    try:
        x = next(it)
    except StopIteration:
        break
    print("item", x)

10 20 30
exhausted
item 10
item 20
item 30


In [2]:
# Iterators are single-use; iterables can be re-iterated
squares = map(lambda v: v * v, range(4))        # an iterator (lazy)
print(list(squares), list(squares))             # second time: empty!
print(iter(numbers) is numbers, iter(it) is it)

[0, 1, 4, 9] []
False True


In [3]:
# A class-based iterator
class Countdown:
    """Iterate n, n-1, ..., 1."""

    def __init__(self, n):
        self.n = n

    def __iter__(self):
        return self

    def __next__(self):
        if self.n <= 0:
            raise StopIteration
        value = self.n
        self.n -= 1
        return value

print(list(Countdown(5)), sum(Countdown(100)))

[5, 4, 3, 2, 1] 5050


In [4]:
# Better: make the *iterable* hand out fresh iterators, so it can be re-used
class Range2:
    def __init__(self, start, stop):
        self.start, self.stop = start, stop

    def __iter__(self):
        current = self.start
        while current < self.stop:
            yield current                 # __iter__ as a generator function
            current += 1

r = Range2(3, 7)
print(list(r), list(r))

[3, 4, 5, 6] [3, 4, 5, 6]


In [5]:
# Two-argument iter(callable, sentinel) and file iteration
import io
stream = io.StringIO("alpha\nbeta\n\ngamma\n")
for line in iter(stream.readline, ""):         # until readline returns ''
    print(repr(line))

'alpha\n'
'beta\n'
'\n'
'gamma\n'


## 2. Generators

A function with `yield` returns a **generator object** (an iterator). Execution *pauses* at each `yield`
and resumes on the next `next()`, keeping local state. Benefits: **lazy** evaluation, O(1) memory,
infinite sequences, readable pipelines.

In [6]:
def countdown(n):
    print("start")
    while n > 0:
        yield n
        n -= 1
    print("done")

gen = countdown(3)
print(type(gen))
print(next(gen))            # runs until the first yield
print(next(gen))
print(list(gen))            # consumes the rest, prints "done"

<class 'generator'>
start
3
2
done
[1]


In [7]:
def fibonacci():
    a, b = 0, 1
    while True:                      # infinite!
        yield a
        a, b = b, a + b

from itertools import islice, takewhile
print(list(islice(fibonacci(), 12)))
print(list(takewhile(lambda v: v < 100, fibonacci())))

[0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89]
[0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89]


In [8]:
# Generator expressions: like list comprehensions with () - lazy
import sys
big_list = [i * i for i in range(100_000)]
big_gen = (i * i for i in range(100_000))
print(sys.getsizeof(big_list), "bytes vs", sys.getsizeof(big_gen), "bytes")
print(sum(i * i for i in range(10)))      # parentheses of sum() double as the generator's

800984 bytes vs 200 bytes
285


In [9]:
# yield from: delegate to a sub-iterator
def flatten(nested):
    for item in nested:
        if isinstance(item, (list, tuple)):
            yield from flatten(item)
        else:
            yield item

print(list(flatten([1, [2, [3, 4]], (5, [6]), 7])))

[1, 2, 3, 4, 5, 6, 7]


In [10]:
# send(): generators can also *receive* values (coroutine-style)
def running_average():
    total, count, avg = 0.0, 0, None
    while True:
        value = yield avg
        total += value
        count += 1
        avg = total / count

ra = running_average()
next(ra)                                  # prime it
print(ra.send(10), ra.send(20), ra.send(60))

10.0 15.0 30.0


### Lazy pipelines
Chain generators so that data flows one item at a time - perfect for big files (e.g. the DNA
challenge in the bonus slides).

In [11]:
def read_lines(text):
    for line in text.splitlines():
        yield line

def non_empty(lines):
    return (ln for ln in lines if ln.strip())

def parse_ints(lines):
    for ln in lines:
        try:
            yield int(ln)
        except ValueError:
            continue

raw = "10\n\n20\nnot a number\n30\n"
print(sum(parse_ints(non_empty(read_lines(raw)))))

60


In [12]:
# Useful itertools companions
import itertools as it
def chunks(iterable, size):
    iterator = iter(iterable)
    while batch := list(it.islice(iterator, size)):
        yield batch

print(list(chunks(range(10), 4)))
print(list(it.pairwise([1, 4, 9, 16])))        # 3.10+: sliding window of 2

[[0, 1, 2, 3], [4, 5, 6, 7], [8, 9]]
[(1, 4), (4, 9), (9, 16)]


## 3. Decorators

Functions are objects, so a function can take a function and return a new one. The decorator syntax

```python
@mydecorator
def function(): ...
```
is exactly `function = mydecorator(function)`.

Why: remove repeated code, separate concerns (logging, timing, caching, auth, retries) from business
logic, keep behaviour explicit and reusable.

In [13]:
def shout(func):
    def wrapper(*args, **kwargs):          # accept anything the wrapped function accepts
        result = func(*args, **kwargs)
        return result.upper()
    return wrapper

@shout
def greet(name):
    """Return a greeting."""
    return f"hello, {name}"

print(greet("ada"))
print(greet.__name__, "|", greet.__doc__)      # metadata of the wrapper leaked through!

HELLO, ADA
wrapper | None


In [14]:
# Fix with functools.wraps: copies __name__, __doc__, __wrapped__, ...
from functools import wraps

def shout(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs).upper()
    return wrapper

@shout
def greet(name):
    """Return a greeting."""
    return f"hello, {name}"

print(greet("ada"), greet.__name__, "|", greet.__doc__)

HELLO, ADA greet | Return a greeting.


In [15]:
# The slides' "debug" decorator
debugging = True

def debug(f):
    @wraps(f)
    def my_wrapper(*args, **kwargs):
        call_string = f"{f.__name__} called with args={args!r}, kwargs={kwargs!r} "
        ret_val = f(*args, **kwargs)
        call_string += "-> " + repr(ret_val)
        if debugging:
            print(call_string)
        return ret_val
    return my_wrapper

@debug
def recursive(k):
    return k * recursive(k - 1) if k > 1 else 1

print(recursive(3))

recursive called with args=(1,), kwargs={} -> 1
recursive called with args=(2,), kwargs={} -> 2
recursive called with args=(3,), kwargs={} -> 6
6


In [16]:
# Stacking order: the decorator closest to the function is applied FIRST
def hello_solar_system(original_function):
    def new_function():
        original_function()
        print("Hello, solar system!")
    return new_function

def hello_galaxy(original_function):
    def new_function():
        original_function()
        print("Hello, galaxy!")
    return new_function

@hello_galaxy
@hello_solar_system
def hello():
    print("Hello, world!")

hello()          # world -> solar system -> galaxy
# equivalent to: hello = hello_galaxy(hello_solar_system(hello))

Hello, world!
Hello, solar system!
Hello, galaxy!


### Decorators with arguments
Add one more layer: a *decorator factory* returns the actual decorator.

In [17]:
import time

def retry(times=3, exceptions=(ConnectionError,), delay=0.0):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, times + 1):
                try:
                    return func(*args, **kwargs)
                except exceptions as err:
                    print(f"  attempt {attempt}/{times} failed: {err}")
                    if attempt == times:
                        raise
                    time.sleep(delay)
        return wrapper
    return decorator

calls = {"n": 0}

@retry(times=3)
def flaky():
    calls["n"] += 1
    if calls["n"] < 3:
        raise ConnectionError("network down")
    return "connected"

print(flaky())

  attempt 1/3 failed: network down
  attempt 2/3 failed: network down
connected


In [18]:
def timed(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        try:
            return func(*args, **kwargs)
        finally:
            elapsed = time.perf_counter() - start
            print(f"{func.__name__} took {elapsed * 1000:.2f} ms")
    return wrapper

@timed
def slow_sum(n):
    return sum(i * i for i in range(n))

slow_sum(200_000);

slow_sum took 10.10 ms


In [19]:
# Class-based decorator: keeps state in the instance
class CountCalls:
    def __init__(self, func):
        wraps(func)(self)
        self.func = func
        self.calls = 0

    def __call__(self, *args, **kwargs):
        self.calls += 1
        return self.func(*args, **kwargs)

@CountCalls
def add(a, b):
    return a + b

add(1, 2); add(3, 4)
print(add.calls, add.__name__)

2 add


In [20]:
# Standard-library decorators you should know
from functools import lru_cache, cache, cached_property, singledispatch, total_ordering

@lru_cache(maxsize=128)
def fib(n):
    return n if n < 2 else fib(n - 1) + fib(n - 2)

print(fib(90), fib.cache_info())

@singledispatch
def describe(x):
    return f"object {x!r}"

@describe.register
def _(x: int):
    return f"int {x}"

@describe.register
def _(x: list):
    return f"list of {len(x)}"

print(describe(3), describe([1, 2]), describe("s"))

2880067194370816120 CacheInfo(hits=88, misses=91, maxsize=128, currsize=91)
int 3 list of 2 object 's'


## 4. Type annotations

Hints are **not enforced at runtime**; they document intent and let tools (`mypy`, `pyright`, IDEs)
find bugs before you run the code. They are also used at runtime by libraries such as `dataclasses`,
`pydantic` and `FastAPI`.

In [21]:
from collections.abc import Callable, Iterable, Iterator, Sequence
from typing import Any, Optional, TypeVar, Union, Literal, TypedDict, Protocol, Final

def mean(values: Sequence[float]) -> float:
    return sum(values) / len(values)

def first_or_none(items: list[int]) -> int | None:       # 3.10+: X | Y == Union[X, Y]
    return items[0] if items else None

def apply(f: Callable[[int], int], xs: Iterable[int]) -> list[int]:
    return [f(x) for x in xs]

def lines(n: int) -> Iterator[str]:                       # a generator's type
    for i in range(n):
        yield f"line {i}"

Mode = Literal["r", "w"]
MAX_RETRIES: Final = 3

print(mean([1, 2, 3]), first_or_none([]), apply(abs, [-1, 2]), list(lines(2)))
print(mean.__annotations__)

2.0 None [1, 2] ['line 0', 'line 1']
{'values': collections.abc.Sequence[float], 'return': <class 'float'>}


In [22]:
# Generics with TypeVar: "returns the same type it receives"
T = TypeVar("T")

def first(items: Sequence[T]) -> T:
    return items[0]

print(first([1, 2]), first("xyz"))

# Python 3.12 syntax:  def first[T](items: Sequence[T]) -> T: ...

class Movie(TypedDict):
    title: str
    year: int

m: Movie = {"title": "Alien", "year": 1979}

class SupportsClose(Protocol):           # structural typing ("duck typing" for type checkers)
    def close(self) -> None: ...

def shutdown(resource: SupportsClose) -> None:
    resource.close()

shutdown(io.StringIO())                  # StringIO has .close(), so it fits without inheriting anything

1 x


### Checking types with `mypy`
```bash
python -m mypy my_module.py              # report inconsistencies
python -m mypy --strict my_module.py     # very strict
```
Example of a bug mypy catches *without running the code*:

```python
def total(prices: list[float]) -> float:
    return sum(prices)

total("12.5")                  # error: Argument 1 has incompatible type "str"; expected "list[float]"
```

Start gradually: annotate function signatures first, add `--ignore-missing-imports` for untyped libs.

## Exercises

**A. `grouper`.** Write a generator `windows(seq, size)` yielding overlapping windows
(`windows([1,2,3,4], 2)` -> `[1,2] [2,3] [3,4]`). Then a *lazy* moving average built on top.

**B. `memoize` from scratch.** Implement your own `memoize` decorator using a `dict` keyed by the arguments,
apply it to a recursive Fibonacci, and compare timings with `@timed`.

**C. `validate_types`.** Write a decorator that reads `func.__annotations__` and raises `TypeError` if an
argument of a simple annotated type (`int`, `str`, ...) has the wrong type. (Hint: `inspect.signature`.)

**D. Streaming word counter.** Build a generator pipeline that reads a big text *line by line*, yields
words, and feeds a `Counter` - without ever holding the whole file in memory.

In [23]:
# Your turn